# Biomarker S8 — Holdout một lần như báo cáo, không dùng k-fold

**Vì sao có notebook này bên cạnh S7.** S7 dùng cross-validation 15 fold, cho con số trung thực nhất
nhưng **mỗi ca được dự đoán bởi một mô hình khác nhau**, và không tồn tại "mô hình cuối cùng" nào để
đem đi làm việc khác. Nhiều ý tưởng tiếp theo lại cần đúng hai thứ mà k-fold không cho: *một* mô hình
đã huấn luyện, và *một* tập test cố định. S8 sinh ra cho việc đó.

**Chia dữ liệu giống hệt báo cáo 13/9:** `GroupShuffleSplit(test_size=0.2, random_state=42)` theo
`subject`. Cỡ tập test rơi vào khoảng 246 ca, đúng con số trong báo cáo, nên mọi bảng ở đây so trực
tiếp được với Bảng 5 và Bảng 6.

**Bảy kiến trúc** — sáu cái của S7, cộng **F** là cái mới:

| | Model | Đầu dự đoán |
|---|---|---|
| A | XGB softmax (= S4 gốc) | — |
| A2 | MLP softmax, đối chứng cho D | Linear |
| B | XGB Frank & Hall | — |
| C | XGB hồi quy + điểm cắt tối ưu QWK | — |
| D | MLP loss ngưỡng + L_mono | Linear |
| E | MLP đủ 4 thành phần loss của slide | Linear |
| **F** | **như E, nhưng đầu dự đoán là MLP** | **MLP 32→32→out** |

F khác E **đúng một chỗ**: mỗi head là một MLP hai lớp thay vì một lớp Linear. Thân giữ nguyên
64 rồi 32, số epoch và seed giữ nguyên. Nên `F − E` là đóng góp riêng của việc làm đầu dự đoán sâu hơn.

> **S8 không thay thế S7.** Kappa của một lần chia dao động khoảng ±0.055, gấp bốn lần con số gộp
> out-of-fold của S7. Dùng S7 khi cần kết luận, dùng S8 khi cần một mô hình cố định.

## 0) Môi trường

In [ ]:
# ============================================================
# Moi dinh nghia nam trong bsc/*.py - notebook chi goi.
# ============================================================
!pip install -q xgboost scikit-learn pandas matplotlib 2>/dev/null
from google.colab import drive
drive.mount("/content/drive")

REPO_URL, REPO_DIR = "https://github.com/AIVIETNAM-AIO-Tuan/bsCart-net.git", "/content/repo"
import os, sys
if not os.path.isdir(f"{REPO_DIR}/bsc"):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
for _m in [k for k in list(sys.modules) if k == "bsc" or k.startswith("bsc.")]:
    del sys.modules[_m]

import torch
from bsc import ordinal as ORD
from bsc.biomarkers import LEGACY_PREFIXES, SURFACE_PREFIXES
print("bsc.ordinal:", ORD.__file__, "| torch", torch.__version__)

## 1) Nạp bảng, tách 96 ca external, chọn feature set

Giống hệt S7. Nhãn tách external **luôn lấy lại từ manifest**, không tin cột trong bảng đặc trưng,
vì bảng S3 được sinh trước khi phép tách tồn tại.

In [ ]:
from pathlib import Path
import json, time
import numpy as np, pandas as pd

BIOM_DIR = Path("/content/drive/MyDrive/OAI_seg/knee_biomarkers_09_09")
V2_CSV = BIOM_DIR / "s6_fcl" / "biomarker_table_v2.csv"
S3_CSV = BIOM_DIR / "biomarker_table.csv"
COMBINED_CSV = BIOM_DIR / "combined_features_table.csv"
OUT_DIR = BIOM_DIR / "s8_holdout"                     # MOI - khong dung chung voi s7_ordinal
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert str(OUT_DIR).startswith("/content/drive/"), "Drive-first"

SRC = V2_CSV if V2_CSV.exists() else S3_CSV
assert SRC.exists(), f"khong thay {V2_CSV} lan {S3_CSV}"
df_all = pd.read_csv(SRC).drop_duplicates("case_id").reset_index(drop=True)
df_all = df_all.dropna(subset=["KL", "subject"]).copy()
df_all["KL"] = df_all["KL"].astype(int)

MANIFEST = BIOM_DIR / "cohort_manifest.csv"
SPLIT_COLS = ["source_dataset", "oaizib_split"]
man = pd.read_csv(MANIFEST).drop_duplicates("case_id")
have = [c for c in SPLIT_COLS if c in man.columns]
assert have, f"{MANIFEST.name} khong co cot danh dau external"
df_all = df_all.drop(columns=[c for c in have if c in df_all.columns], errors="ignore")
df_all = df_all.merge(man[["case_id"] + have], on="case_id", how="left")

is_ext = pd.Series(False, index=df_all.index)
if "source_dataset" in have:
    is_ext |= df_all["source_dataset"].astype(str).str.startswith("reserved_external_validation")
if "oaizib_split" in have:
    is_ext |= df_all["oaizib_split"].astype(str).str.contains("ext", case=False, na=False)
df = df_all[~is_ext].reset_index(drop=True)
print(f"tong {len(df_all)} | dung cho S8: {len(df)} | external giu rieng: {int(is_ext.sum())}")

legacy_cols = [c for c in df.columns if c.startswith(LEGACY_PREFIXES)]
surface_cols = [c for c in df.columns if c.startswith(SURFACE_PREFIXES)]
FEATURE_SETS = {"legacy_s3": legacy_cols}
if surface_cols:
    FEATURE_SETS["s6_all"] = legacy_cols + surface_cols
if COMBINED_CSV.exists():
    comb = pd.read_csv(COMBINED_CSV, low_memory=False).drop_duplicates("case_id")
    rad_cols = [c for c in comb.columns if c.startswith("rad_")]
    df = df.merge(comb[["case_id"] + rad_cols], on="case_id", how="left")
    FEATURE_SETS["s5_radiomics"] = legacy_cols + rad_cols
    if surface_cols:
        FEATURE_SETS["s6_all_plus_radiomics"] = legacy_cols + surface_cols + rad_cols
    print(f"nap {len(rad_cols)} cot radiomics")

CLASSES = sorted(df["KL"].unique().tolist())
K = len(CLASSES)
y_idx = df["KL"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy(np.int64)
groups = df["subject"].astype(str).to_numpy()
oa_target = (df["KL"] >= 2).to_numpy(np.float32)
print("so feature:", {k: len(v) for k, v in FEATURE_SETS.items()})
print("KL:", CLASSES, "| n moi lop:", np.bincount(y_idx, minlength=K).tolist())

## 2) Chia holdout **đúng như báo cáo** — một lần, theo subject

`GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)`. Cell in ra cỡ mẫu và phân bố lớp hai
phía để bạn đối chiếu với Bảng 2 và các ma trận nhầm lẫn trong báo cáo.

Chia theo `subject` chứ không theo ca: một người có thể có hai chân hoặc hai lần khám, chia theo ca
thì cùng một đầu gối lọt cả hai phía và mô hình chỉ cần nhớ mặt là xong.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
TR, TE = next(gss.split(df, y_idx, groups))
assert not (set(groups[TR]) & set(groups[TE])), "ro ri subject giua train va test"

df["split"] = np.where(np.isin(np.arange(len(df)), TE), "test", "train")
print(f"train {len(TR)} ca / {len(set(groups[TR]))} subject | "
      f"test {len(TE)} ca / {len(set(groups[TE]))} subject")
print("bao cao 13/9 co n_test = 246")

dist = pd.DataFrame({"train": np.bincount(y_idx[TR], minlength=K),
                     "test": np.bincount(y_idx[TE], minlength=K)},
                    index=[f"KL{c}" for c in CLASSES])
dist["test_%"] = (dist["test"] / dist["test"].sum() * 100).round(1)
display(dist)

# Ghim lai ngay: moi thu xay tiep phai DOC file nay thay vi chia lai, neu khong hai thi
# nghiem se khong so duoc voi nhau.
df[["case_id", "subject", "KL", "split"]].to_csv(OUT_DIR / "holdout_split.csv", index=False)
print("da ghim split vao holdout_split.csv")

## 3) Bảy model — cùng train, cùng test

Siêu tham số giữ nguyên của S7 để so được. Chọn đặc trưng chạy **một lần trên tập train**, đúng theo
logic holdout: không có fold nào để chạy lại.

In [ ]:
from xgboost import XGBClassifier, XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_predict

XGB_KW = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
              colsample_bytree=0.8, random_state=42, n_jobs=2)
MLP_EPOCHS = 400
HEAD_HIDDEN_F = (32,)      # F: moi head la MLP 32 -> 32 -> out, thay vi mot Linear

def m_xgb_softmax(Xtr, ytr, Xte, **_):
    clf = XGBClassifier(objective="multi:softprob", num_class=K, eval_metric="mlogloss", **XGB_KW)
    clf.fit(Xtr, ytr)
    return dict(y_pred=clf.predict_proba(Xte).argmax(1), model=clf)

def m_xgb_frankhall(Xtr, ytr, Xte, **_):
    fh = ORD.FrankHall(lambda: XGBClassifier(objective="binary:logistic",
                                             eval_metric="logloss", **XGB_KW))
    fh.fit(Xtr, ytr, K)
    p = fh.predict_proba_thresholds(Xte)
    return dict(y_pred=ORD.decode_count(p), p_thr=p, y_alt=ORD.decode_cumdiff(p), model=fh)

def m_xgb_reg(Xtr, ytr, Xte, gtr=None, **_):
    reg = XGBRegressor(objective="reg:squarederror", **XGB_KW)
    inner = cross_val_predict(reg, Xtr, ytr, groups=gtr, cv=GroupKFold(n_splits=4))
    cuts = ORD.fit_cutpoints(inner, ytr, K)
    reg.fit(Xtr, ytr)
    return dict(y_pred=ORD.apply_cutpoints(reg.predict(Xte), cuts), cuts=cuts, model=reg)

def _mlp_runner(lambdas, decode, head_hidden=()):
    def run(Xtr, ytr, Xte, oa_tr=None, **_):
        model, info = ORD.train_ordinal_mlp(Xtr, ytr, K, oa_target=oa_tr, lambdas=lambdas,
                                            epochs=MLP_EPOCHS, seed=0, head_hidden=head_hidden)
        out = ORD.predict_ordinal_mlp(model, info, Xte)
        r = dict(y_pred=out[decode], model=model, info=info)
        if lambdas.get("ord", 0) > 0:
            r["p_thr"] = out["p_thr"]
            r["y_alt"] = out["y_softmax"] if lambdas.get("cls", 0) > 0 else out["y_cumdiff"]
        return r
    return run

MODELS = {
    "A_xgb_softmax":       m_xgb_softmax,
    "A2_mlp_softmax":      _mlp_runner(ORD.SOFTMAX_ONLY_LAMBDAS, "y_softmax"),
    "B_xgb_frankhall":     m_xgb_frankhall,
    "C_xgb_reg_cutpoints": m_xgb_reg,
    "D_mlp_ordinal_only":  _mlp_runner(ORD.ORDINAL_ONLY_LAMBDAS, "y_count"),
    "E_mlp_slide":         _mlp_runner(ORD.SLIDE_LAMBDAS, "y_count"),
    "F_mlp_slide_mlphead": _mlp_runner(ORD.SLIDE_LAMBDAS, "y_count", HEAD_HIDDEN_F),
}
print("so model:", len(MODELS))

## 4) Chạy — một lần train, một lần test

In [ ]:
SELECT_ABOVE = 120

rows, PRED, FITTED = [], {}, {}
t0 = time.time()
for fs_name, cols in FEATURE_SETS.items():
    X_all = df[cols].to_numpy(np.float32)
    imp = SimpleImputer(strategy="median").fit(X_all[TR])        # fit CHI tren train
    Xtr, Xte = imp.transform(X_all[TR]), imp.transform(X_all[TE])
    if len(cols) > SELECT_ABOVE:
        keep = ORD.select_features(Xtr, y_idx[TR], seed=0)        # fit CHI tren train
        Xtr, Xte = Xtr[:, keep], Xte[:, keep]
        print(f"{fs_name}: {len(cols)} -> {len(keep)} cot sau khi chon")
    for m_name, fn in MODELS.items():
        r = fn(Xtr, y_idx[TR], Xte, gtr=groups[TR], oa_tr=oa_target[TR])
        yp = np.asarray(r["y_pred"], np.int64)
        PRED[(fs_name, m_name)] = yp
        FITTED[(fs_name, m_name)] = r
        rows.append(dict(feature_set=fs_name, model=m_name, n_test=len(TE),
                         qwk=ORD.qwk(y_idx[TE], yp, K),
                         acc=float((yp == y_idx[TE]).mean()),
                         mae=ORD.mae(y_idx[TE], yp),
                         off_by_2=ORD.off_by_rate(y_idx[TE], yp, 2)))
res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / "holdout_metrics.csv", index=False)
print(f"xong trong {(time.time() - t0) / 60:.1f} phut")

## 5) Kết quả — định dạng như Bảng 5 của báo cáo

`off_by_2` là tỉ lệ ca lệch từ 2 bậc KL trở lên, chính là cột mà báo cáo gọi là lỗi lâm sàng nghiêm
trọng nhất. Báo cáo đo được 22.0% cho S4 gốc, 9.3% cho nominal 93 cột, 7.3% cho Frank-Hall và 6.5%
cho Regression.

In [ ]:
import matplotlib.pyplot as plt

for fs in FEATURE_SETS:
    sub = res[res.feature_set == fs].set_index("model")[["acc", "qwk", "mae", "off_by_2"]]
    print(f"--- {fs} (n_test={len(TE)})")
    display(sub.assign(acc=lambda d: (d.acc * 100).round(1),
                       off_by_2=lambda d: (d.off_by_2 * 100).round(1)).round(3))

fs0 = list(FEATURE_SETS)[-1]
sub = res[res.feature_set == fs0].set_index("model")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sub["qwk"].plot.bar(ax=ax[0], rot=25, color="#0E6F6E")
ax[0].set_ylabel("QWK"); ax[0].set_title(fs0 + " - cao hon la tot hon")
(sub["off_by_2"] * 100).plot.bar(ax=ax[1], rot=25, color="#BC2A30")
ax[1].set_ylabel("off-by>=2 (%)"); ax[1].set_title("thap hon la tot hon")
plt.tight_layout(); plt.savefig(OUT_DIR / "holdout_metrics.png", dpi=130); plt.show()

## 6) Precision / Recall / F1 theo lớp + ma trận nhầm lẫn

So trực tiếp được với Bảng 6 của báo cáo, vì cùng cỡ tập test và cùng cách chia.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

rows = []
for (fs, m_name), yp in PRED.items():
    pr, rc, f1, sup = precision_recall_fscore_support(
        y_idx[TE], yp, labels=list(range(K)), zero_division=0)
    for k in range(K):
        rows.append(dict(feature_set=fs, model=m_name, KL=str(CLASSES[k]), n=int(sup[k]),
                         precision=pr[k], recall=rc[k], f1=f1[k]))
    rows.append(dict(feature_set=fs, model=m_name, KL="macro", n=int(sup.sum()),
                     precision=pr.mean(), recall=rc.mean(), f1=f1.mean()))
per_class = pd.DataFrame(rows)
per_class.to_csv(OUT_DIR / "holdout_per_class.csv", index=False)

FS_SHOW = list(FEATURE_SETS)[-1]
sub = per_class[per_class.feature_set == FS_SHOW]
print("feature set dang hien:", FS_SHOW)
for metric in ["recall", "precision", "f1"]:
    print()
    print(metric.upper() + " theo lop (%)")
    display((sub.pivot(index="model", columns="KL", values=metric) * 100).round(1))

for m_name in ["A_xgb_softmax", "B_xgb_frankhall", "F_mlp_slide_mlphead"]:
    if (FS_SHOW, m_name) in PRED:
        print()
        print("confusion - " + m_name + " (hang = that, cot = doan)")
        print(confusion_matrix(y_idx[TE], PRED[(FS_SHOW, m_name)], labels=list(range(K))))

## 7) F so với E — đầu dự đoán sâu hơn có giúp không

`F − E` là đóng góp riêng của việc đổi head từ Linear sang MLP, vì mọi thứ khác giữ nguyên. Nếu chênh
lệch nhỏ hơn dao động của một lần chia, khoảng ±0.055 QWK, thì chưa kết luận được gì.

In [ ]:
cmp = res.pivot(index="feature_set", columns="model", values="qwk")
if {"E_mlp_slide", "F_mlp_slide_mlphead"} <= set(cmp.columns):
    out = pd.DataFrame({"E (head Linear)": cmp["E_mlp_slide"],
                        "F (head MLP)": cmp["F_mlp_slide_mlphead"]})
    out["F - E"] = out["F (head MLP)"] - out["E (head Linear)"]
    display(out.round(3))
    d = out["F - E"].abs().max()
    print(f"chenh lech lon nhat {d:.3f} | dao dong cua mot lan chia ~0.055 "
          f"-> {'CHUA ket luan duoc' if d < 0.055 else 'dang chu y'}")

## 8) Lưu model và cấu hình

In [ ]:
import pickle

with open(OUT_DIR / "fitted_models.pkl", "wb") as f:
    pickle.dump({f"{fs}__{m}": FITTED[(fs, m)].get("model") for fs, m in FITTED}, f)
pred_out = df[["case_id", "subject", "KL", "split"]].copy()
for (fs, m_name), yp in PRED.items():
    col = np.full(len(df), np.nan)
    col[TE] = [CLASSES[i] for i in yp]
    pred_out[f"pred_{fs}__{m_name}"] = col
pred_out.to_csv(OUT_DIR / "holdout_predictions.csv", index=False)

json.dump(dict(source=str(SRC), classes=CLASSES, n=len(df), n_train=len(TR), n_test=len(TE),
               split="GroupShuffleSplit(test_size=0.2, random_state=42) theo subject",
               xgb=XGB_KW, mlp_epochs=MLP_EPOCHS, head_hidden_F=list(HEAD_HIDDEN_F),
               feature_sets={k: len(v) for k, v in FEATURE_SETS.items()}),
          open(OUT_DIR / "run_config.json", "w"), indent=2)
print("da luu vao", OUT_DIR)

## Ghi chú

- **S8 không thay thế S7.** Con số của một lần chia dao động khoảng ±0.055 QWK, gấp bốn lần con số gộp
  out-of-fold của S7. Dùng S7 để kết luận, dùng S8 khi cần một mô hình cố định và một tập test cố định.
- `holdout_split.csv` ghim lại ca nào train ca nào test. Mọi thứ xây tiếp nên **đọc file đó** thay vì
  chia lại, nếu không hai thí nghiệm sẽ không so được với nhau.
- F khác E đúng một chỗ là head sâu hơn. Nếu F không hơn E thì đó là một kết luận đáng ghi: trên bảng
  biomarker, làm đầu dự đoán sâu hơn không giúp gì.
- Chưa làm ở đây: **trích latent vector**. Ý tưởng đó đến từ mô hình phân loại ảnh, mà nhánh ảnh thì
  chưa có. Khi nào có encoder ảnh thì mới bàn tiếp.